In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

# 크롤링할 ACL 2024 학회 페이지 URL
acl_url = "https://aclanthology.org/events/acl-2024/#2024acl-short"

# 웹페이지 요청
response = requests.get(acl_url)

# 응답 상태 확인
if response.status_code == 200:
    # HTML 파싱
    soup = BeautifulSoup(response.text, "html.parser")

    # "Proceedings of the 62nd Annual Meeting of the Association for Computational Linguistics (Volume 1: Long Papers)" 찾기
    target_section = soup.find("h2", string="Proceedings of the 62nd Annual Meeting of the Association for Computational Linguistics (Volume 1: Long Papers)")

    if target_section:
        # 해당 섹션 아래의 논문 리스트 찾기
        paper_list = target_section.find_next("div").find_all("p")

        papers = []
        for paper in paper_list:
            title_tag = paper.find("a")
            authors_tag = paper.find("i")
            
            if title_tag:
                title = title_tag.get_text(strip=True)
                link = title_tag["href"]
            else:
                title = "Unknown"
                link = "N/A"

            authors = authors_tag.get_text(strip=True) if authors_tag else "Unknown"

            papers.append({"Title": title, "Authors": authors, "Link": f"https://aclanthology.org{link}"})

        # DataFrame 생성
        df = pd.DataFrame(papers)

        # CSV로 저장
        csv_file_path = "acl_2024_long_papers.csv"
        df.to_csv(csv_file_path, index=False)

        print(f"✅ 논문 리스트가 저장됨: {csv_file_path}")

    else:
        print("❌ 원하는 섹션을 찾을 수 없음.")
else:
    print(f"❌ 페이지 요청 실패: {response.status_code}")
